### Decoder Only Transformer from scratch 

In [1]:
import tensorflow as tf
tf.__version__

2026-04-10 11:14:00.670000: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-10 11:14:01.542010: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-10 11:14:05.307518: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


'2.20.0'

In [2]:
# Attention Mechanism
def scaled_dot_product_attention(q, k, v, mask=None):
   
    #QK^T
    matmul_qk = tf.linalg.matmul(q, k, transpose_b=True)

    #Scaling
    dk = tf.cast(tf.shape(k)[-1], tf.float32)
    scaled_logits = matmul_qk / tf.math.sqrt(dk)

    #Masking
    if mask is not None:
        mask = tf.cast(mask, tf.float32)
        scaled_logits += (mask * -1e9)

    # 4.Softmax Normalization
    attention_weights = tf.nn.softmax(scaled_logits, axis=-1)

    # 5. Multiply with V
    output = tf.linalg.matmul(attention_weights, v)

    return output, attention_weights

In [3]:
#Multihead Attention

class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.depth = d_model // num_heads

        # Linear projections
        self.wq = tf.keras.layers.Dense(d_model)
        self.wk = tf.keras.layers.Dense(d_model)
        self.wv = tf.keras.layers.Dense(d_model)

        # Final output projection
        self.dense = tf.keras.layers.Dense(d_model)

    def split_heads(self, x, batch_size):
        """
        Input shape:  (batch, seq_len, d_model)
        Output shape: (batch, num_heads, seq_len, depth)
        """
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, x, mask=None):
        batch_size = tf.shape(x)[0]

        # 1. Linear projections
        q = self.wq(x)  # (B, S, d_model)
        k = self.wk(x)
        v = self.wv(x)

        # 2. Split into heads
        q = self.split_heads(q, batch_size)  # (B, H, S, depth)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)

        # 3. Scaled dot-product attention
        scaled_attention,attention_weights =scaled_dot_product_attention(q,k,v,mask)

        # 4. Transpose back
        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])
        # (B, S, H, depth)

        # 5. Concatenate heads
        concat_attention=tf.reshape(scaled_attention,(batch_size,-1,self.d_model))
        # (B, S, d_model)

        # 6. Final linear layer
        output = self.dense(concat_attention)

        return output, attention_weights

In [4]:
# checking for shapes of input and output
batch_size = 2
seq_len = 5
d_model = 128
num_heads = 4

x = tf.random.uniform((batch_size, seq_len, d_model))

mha = MultiHeadAttention(d_model, num_heads)

output, weights = mha(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Attention weights shape:", weights.shape)

Input shape: (2, 5, 128)
Output shape: (2, 5, 128)
Attention weights shape: (2, 4, 5, 5)


2026-04-10 11:14:15.794948: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
